# Orbit Wars Set Transformer Modeli

Bu defter `model.py` dosyasındaki `OrbitWarsPolicy` ağını hem teorik hem de pratik açıdan açıklar. Amaç sadece kodun ne yaptığını göstermek değil; neden Set Transformer seçildiğini, attention bias ve mask mekanizmalarının oyundaki karar problemine nasıl bağlandığını da netleştirmektir.

Modelin özeti:

- Girdi, sabit uzunluklu ama maskelenmiş bir token setidir: gezegenler, filolar ve bir global token.
- Encoder, self-attention kullanan bir Set Transformer benzeridir.
- Pairwise geometri `edge_bias` ile attention skorlarına doğrudan eklenir.
- Actor, aksiyonu üç parçaya ayırır: kaynak gezegen, hedef gezegen, gönderilecek gemi oranı.
- Critic, global token üzerinden durum değerini üretir.

## 1. Problem: Orbit Wars Durumu Neden Set Gibi?

Orbit Wars gözlemi doğal olarak sabit uzunluklu bir vektör değildir. Her turda farklı sayıda gezegen ve havada farklı sayıda filo olabilir. Ayrıca gezegenlerin listedeki sırası semantik olarak önemli değildir: aynı gezegen kümesini farklı sırada vermek, oyunun stratejik anlamını değiştirmez.

Bu yüzden model şu fikre yaslanır:

$$X = \{x_1, x_2, \ldots, x_T\}$$

Burada her $x_i$ bir token temsilidir. Tokenlar üç tiptir:

| Token tipi | Kaynak | Anlam |
| --- | --- | --- |
| Planet token | `planet_feats` | Bir gezegenin sahiplik, gemi, üretim, konum ve hareket bilgisi |
| Fleet token | `fleet_feats` | Havadaki bir filonun sahiplik, gemi, hız, yön ve ETA bilgisi |
| Global token | `global_feats` | Tur, toplam güç, üretim, gelen tehdit gibi global özet |

`model.py` bu token sırasını sabit kabul eder:

```text
[planet_0, ..., planet_{P_max-1}, fleet_0, ..., fleet_{F_max-1}, GLOBAL]
```

Toplam token sayısı:

$$T = P_{max} + F_{max} + 1$$

## 2. Notebook Kurulumu

Aşağıdaki hücre, defter `orbit-wars-data/notebooks/notebooks` altında çalıştırılsa da repo kökünü import path'e ekler.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
while REPO.name != 'Orbit_Wars' and REPO.parent != REPO:
    REPO = REPO.parent

sys.path.insert(0, str(REPO))
print('Repo root:', REPO)

In [ ]:
import math
import torch

from obs_builder import build_observation, decode_action
from model import OrbitWarsPolicy, pack_observations

## 3. Oyuncak Bir Observation

Modeli anlatmak için küçük ve deterministik bir oyun durumu kullanacağız. Burada `p_max=8`, `f_max=4` seçiyoruz; gerçek default değerler `32` ve `32`.

In [ ]:
toy_obs = {
    'planets': [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],
        [1, 0, 80.0, 70.0, 2.0, 50, 2],
        [2, 1, 30.0, 30.0, 2.0, 80, 2],
        [3, -1, 20.0, 80.0, 2.0, 10, 1],
    ],
    'fleets': [
        [10, 1, 60.0, 60.0, 0.0, 2, 25],
    ],
    'initial_planets': [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],
        [1, 0, 80.0, 70.0, 2.0, 50, 2],
        [2, 1, 30.0, 30.0, 2.0, 80, 2],
        [3, -1, 20.0, 80.0, 2.0, 10, 1],
    ],
    'comet_planet_ids': [],
    'comets': [],
}

obs = build_observation(toy_obs, player=0, angular_velocity=0.05, p_max=8, f_max=4)

print('planet_feats:', obs.planet_feats.shape)
print('fleet_feats :', obs.fleet_feats.shape)
print('global_feats:', obs.global_feats.shape)
print('edge_bias   :', obs.edge_bias.shape)
print('planet_mask :', obs.planet_mask.astype(int))
print('fleet_mask  :', obs.fleet_mask.astype(int))
print('source_mask :', obs.source_mask.astype(int))

## 4. Feature Pipeline ve Maskler

`obs_builder.py` modelin beklediği sabit şekilli tensörleri üretir. Buradaki kritik fikir padding + mask yaklaşımıdır.

| Alan | Şekil | Ne işe yarar? |
| --- | --- | --- |
| `planet_feats` | `(P_max, 15)` | Gezegen token ham özellikleri |
| `fleet_feats` | `(F_max, 11)` | Filo token ham özellikleri |
| `global_feats` | `(12,)` | Tüm durumun global özeti |
| `edge_bias` | `(T, T, 3)` | Token çiftleri arası mesafe, dx, dy |
| `planet_mask` | `(P_max,)` | Gerçek gezegen mi, padding mi? |
| `fleet_mask` | `(F_max,)` | Gerçek filo mu, padding mi? |
| `source_mask` | `(P_max,)` | Bu gezegenden filo çıkarılabilir mi? |

Model iki farklı mask fikri kullanır:

- Padding mask: attention içinde padding tokenlara bakılmasını engeller.
- Action mask: invalid kaynak veya hedef aksiyonlara olasılık verilmesini engeller.

Bu ayrım önemli: her padding problemi attention problemi değildir; bazıları policy head seviyesinde çözülür.

## 5. Tokenlaştırma: Heterojen Veriyi Ortak Uzaya Taşımak

`OrbitWarsPolicy._build_tokens` üç farklı feature tipini ortak `d_model` boyutuna projekte eder:

$$z_i = W_{type(i)} x_i + e_{type(i)}$$

Kod karşılığı:

```python
planet_tok = self.planet_proj(planet_feats)
fleet_tok = self.fleet_proj(fleet_feats)
global_tok = self.global_proj(global_feats).unsqueeze(1)
x = torch.cat([planet_tok, fleet_tok, global_tok], dim=1)
x = x + self.type_embed(self.type_ids)[None]
```

Burada type embedding küçük ama önemli bir detaydır. Gezegen ve filo feature vektörleri farklı anlamlara gelir; projeksiyondan sonra ikisi de `d_model` boyutlu olduğu için modele token tipini ayrıca söylemek gerekir.

In [ ]:
batch = pack_observations([obs])
net = OrbitWarsPolicy(p_max=8, f_max=4, d_model=32, n_heads=2, n_layers=2)

with torch.no_grad():
    x0 = net._build_tokens(batch['planet_feats'], batch['fleet_feats'], batch['global_feats'])
    attn_bias = net._build_attn_bias(batch['edge_bias'])
    padding_mask = net._build_padding_mask(batch['planet_mask'], batch['fleet_mask'])

print('tokens      :', tuple(x0.shape), '=(B, T, d_model)')
print('attn_bias  :', tuple(attn_bias.shape), '=(B, heads, T, T)')
print('padding    :', tuple(padding_mask.shape), 'True = PAD')
print('type_ids   :', net.type_ids.tolist())

## 6. Self-Attention Teorisi

Standart multi-head self-attention şu üç projeksiyonu üretir:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Bir tokenın diğer tokenlara ne kadar bakacağını attention skorları belirler:

$$S = \frac{QK^T}{\sqrt{d_h}}$$

Sonra softmax ile ağırlıklar elde edilir:

$$A = \mathrm{softmax}(S)$$

Çıkış:

$$Y = AV$$

Orbit Wars açısından bu şu demektir: bir gezegen tokenı, kendi durumunu güncellerken diğer gezegenlere, düşman filolara ve global duruma bakabilir. Örneğin kaynak gezegen güçlü olsa bile yakındaki düşman filosu veya daha değerli hedef başka bir token olarak attention içinde hesaba katılabilir.

## 7. Edge Bias: Geometriyi Attention'a Katmak

`model.py` standart attention'a bir ekleme yapar:

$$S_{hij} = \frac{q_{hi} \cdot k_{hj}}{\sqrt{d_h}} + b_{hij}$$

`b_{hij}` değeri `edge_bias[i, j] = (dist, dx, dy)` feature'larından öğrenilir:

```python
b = self.edge_proj(edge_bias)          # (B, T, T, H)
b = b.permute(0, 3, 1, 2)              # (B, H, T, T)
scores = scores + attn_bias
```

Bunun pratik etkisi şudur: model mesafe, yön ve yakınlık gibi geometrik ilişkileri sıfırdan koordinatlardan çıkarmak zorunda kalmaz. Attention mekanizmasına doğrudan geometrik önbilgi verilir.

Bu bir GNN edge feature kullanımına benzer; ama mesajlaşma katmanını ayrı bir graph library ile kurmak yerine Transformer attention skorlarına bias olarak eklenir.

In [ ]:
with torch.no_grad():
    # İlk batch, ilk head için token 0'ın tokenlara olan learned bias değerleri.
    head0_from_planet0 = attn_bias[0, 0, 0]

print('planet_0 -> tüm tokenlar için head-0 bias:')
print(head0_from_planet0.round(decimals=3))

print('\nHam edge feature örnekleri:')
print('planet_0 -> planet_1:', batch['edge_bias'][0, 0, 1].tolist())
print('planet_0 -> global  :', batch['edge_bias'][0, 0, -1].tolist())

## 8. Padding Mask: Set Boyutu Değişken, Tensor Boyutu Sabit

Attention içinde padding tokenlar key olarak maskelenir:

```python
mask = key_padding_mask[:, None, None, :]
scores = scores.masked_fill(mask, float('-inf'))
```

Yani gerçek tokenlar padding tokenlara attention veremez. Padding query satırları teknik olarak var kalır; ama onların çıktısı downstream aksiyonlarda zaten maskelendiği için semantik bir karşılık taşımaz.

`torch.nan_to_num` güvenlik için kullanılır. Eğer bir attention satırı tamamen `-inf` olursa softmax `NaN` üretebilir; kod bunu sıfıra çeker.

## 9. TransformerBlock: Pre-Norm Residual Yapı

Her blok şu akışı izler:

$$X' = X + \mathrm{Attention}(\mathrm{LN}(X))$$

$$Y = X' + \mathrm{FFN}(\mathrm{LN}(X'))$$

Bu `pre-norm` tasarımdır. LayerNorm attention/FFN öncesinde uygulanır. Derin Transformer eğitiminde pre-norm genellikle daha stabil gradyan akışı verir.

FFN kısmı iki lineer katman ve GELU aktivasyonundan oluşur:

```python
Linear(d_model, 4*d_model) -> GELU -> Linear(4*d_model, d_model)
```

## 10. Actor: Aksiyonu Parçalayarak Modellemek

Orbit Wars aksiyonu tek bir kategorik seçim değildir. Bir ham env aksiyonu kabaca şudur:

```text
[from_planet_id, angle, ships]
```

Model bunu slot tabanlı üç karara ayırır:

$$p(a|s) = p(src|s)\;p(tgt|src,s)\;p(ship\_bin|src,s)$$

Bu faktörizasyon kombinatorik aksiyon uzayını küçültür. Doğrudan tüm `(src, tgt, ship)` kombinasyonlarına ayrı logit üretmek yerine:

- Kaynak için `(P_max + 1)` logit üretilir. Son slot NO-OP'tur.
- Hedef için her kaynak gezegenden her hedef gezegene pointer skoru üretilir.
- Gemi oranı için her kaynak gezegen başına `n_ship_bins` logit üretilir.

Varsayılan ship bin anlamları `decode_action` tarafında şuna karşılık gelir:

```python
(0.1, 0.25, 0.5, 0.75, 1.0)
```

In [ ]:
with torch.no_grad():
    out = net(**batch)

print('src_logits :', tuple(out.src_logits.shape), 'P_max + NO-OP')
print('tgt_logits :', tuple(out.tgt_logits.shape), 'source x target')
print('ship_logits:', tuple(out.ship_logits.shape), 'source x ship_bin')
print('value      :', tuple(out.value.shape))
print('tokens     :', tuple(out.tokens.shape))

## 11. Source Head ve NO-OP

Kaynak seçimi gezegen tokenları üzerinden yapılır:

```python
src_raw = self.src_head(planet_tok).squeeze(-1)  # (B, P)
src_raw = src_raw.masked_fill(~source_mask, very_neg)
src_logits = torch.cat([src_raw, noop], dim=1)  # (B, P+1)
```

`source_mask` sadece bize ait ve yeterli gemisi olan gezegenleri açık bırakır. Böylece model illegal kaynaklardan filo çıkarma olasılığı öğrenmez; bu olasılık en baştan sıfırlanır.

NO-OP ayrı, öğrenilebilir bir logittir. Bu önemli çünkü bazı durumlarda hiçbir şey yapmamak en iyi aksiyon olabilir.

In [ ]:
src_probs = torch.softmax(out.src_logits, dim=-1)

print('source_mask       :', batch['source_mask'][0].int().tolist())
print('source prob       :', src_probs[0].round(decimals=4).tolist())
print('son eleman NO-OP  :', src_probs[0, -1].item())
print('invalid source sum:', src_probs[0, 2:8].sum().item())

## 12. Target Head: Pointer Attention

Hedef seçimi, kaynak gezegen tokenından hedef gezegen tokenlarına doğru bir pointer skoru gibi hesaplanır:

$$score(src,tgt) = \frac{q_{src} \cdot k_{tgt}}{\sqrt{d_{model}}}$$

Kod:

```python
q = self.tgt_query(planet_tok)
k = self.tgt_key(planet_tok)
tgt_logits = torch.matmul(q, k.transpose(-2, -1)) / sqrt(d_model)
```

Sonra iki maske uygulanır:

- `target == source` yasaktır.
- Padding gezegenler hedef olamaz.

Bu nedenle diagonal ve padding target kolonları `very_neg` yapılır.

In [ ]:
tgt_probs = torch.softmax(out.tgt_logits, dim=-1)

print('source 0 için target olasılıkları:')
print(tgt_probs[0, 0].round(decimals=4).tolist())
print('self target prob  :', tgt_probs[0, 0, 0].item())
print('padding target sum:', tgt_probs[0, 0, 4:].sum().item())

## 13. Ship Head: Sürekli Sayıyı Ayrık Bin'e Çevirmek

Geminin tam sayısını doğrudan regression ile tahmin etmek yerine model ayrık oran bin'i seçer:

```python
ship_logits = self.ship_head(planet_tok)  # (B, P, n_ship_bins)
```

Sonradan `decode_action`, seçilen bin'i kaynak gezegenin mevcut gemi sayısıyla çarpar. Örneğin bin `0.5` ise mevcut gemilerin yaklaşık yarısı gönderilir.

Bu seçim RL için pratik bir avantaj sağlar: policy dağılımı kategorik kalır, log-prob ve entropy hesapları kolaylaşır.

## 14. Critic: Global Token Üzerinden Value

Transformer bloklarından sonra son token global tokendır:

```python
global_tok = x[:, -1]
value = self.value_head(global_tok).squeeze(-1)
```

Global token başlangıçta global feature'lardan gelir; self-attention katmanlarından geçerken tüm gezegen ve filo tokenlarından bilgi toplayabilir. Bu yüzden value head için doğal bir havuzlanmış temsil gibi davranır.

PPO/A2C gibi actor-critic algoritmalarında bu `V(s)` tahminidir.

## 15. `act`: Dağılımdan Aksiyon Örneklemek

`act` metodu `PolicyOutput` içindeki logitlerden kategorik dağılımlar kurar:

1. `src ~ Categorical(src_logits)`
2. Seçilen `src` ile ilgili target satırı alınır.
3. `tgt ~ Categorical(tgt_logits[src])`
4. Seçilen `src` ile ilgili ship-bin satırı alınır.
5. `ship_bin ~ Categorical(ship_logits[src])`

NO-OP seçildiyse target ve ship teknik olarak yine shape tutarlılığı için örneklenir; ama log-prob hesabında target/ship katkısı sıfırlanır:

$$\log p(a|s) = \log p(src|s) + \mathbb{1}_{src \ne NOOP}\left[\log p(tgt|src,s) + \log p(ship|src,s)\right]$$

In [ ]:
action = net.act(out, deterministic=False)
for k, v in action.items():
    print(f'{k:8s}:', v.tolist())

det_action = net.act(out, deterministic=True)
print('\nDeterministik aksiyon:')
for k, v in det_action.items():
    print(f'{k:8s}:', v.tolist())

## 16. Slot'tan Env Aksiyonuna Dönmek

Model gezegen id'si değil slot index'i seçer. `decode_action` slotları tekrar env id'lerine çevirir:

```python
src_id = int(obs.planet_ids[src_slot])
tgt_id = int(obs.planet_ids[tgt_slot])
```

Ardından `navigation.compute_launch` ile gerçek açı hesaplanır. Böylece policy ağı doğrudan fizik/geometri çözmek zorunda kalmaz; stratejik seçimi yapar, deterministik navigation katmanı bunu env aksiyonuna dönüştürür.

Bu ayrım mimari olarak temizdir:

- Model: hangi kaynaktan, hangi hedefe, ne kadar gemi?
- Navigation: hangi açıyla fırlatılırsa hedefe gider?

## 17. Neden Bu Mimari İyi Bir Uyum?

| Orbit Wars ihtiyacı | Modeldeki karşılık |
| --- | --- |
| Değişken sayıda nesne | Padding + mask |
| Sıra bağımsız gezegen/filo listeleri | Self-attention tabanlı set encoder |
| Mesafe ve yön çok önemli | Edge bias ile attention skorlarına geometri |
| Illegal aksiyonlar sık | Source/target mask |
| `from, target, ships` kombinatorik uzayı | Faktörize actor head |
| RL için value tahmini | Global token critic |

Özellikle edge bias tasarımı oyuna iyi oturur. Saf Transformer koordinatlardan mesafeyi öğrenebilir; ama burada mesafe ve yön zaten oyun mekaniğinin merkezinde olduğu için bu bilgiyi attention skoruna doğrudan vermek daha veri verimli bir başlangıç sağlar.

## 18. Eğitimde Kullanım Notları

Bu model PPO/A2C tarzı actor-critic eğitim için doğrudan uygundur:

- `out.value`: critic tahmini `V(s)`
- `action['log_prob']`: policy loss için log olasılık
- `src_logits`, `tgt_logits`, `ship_logits`: entropy bonus için kullanılabilir

Dikkat edilmesi gereken noktalar:

- Target/ship entropy NO-OP satırlarında hesaba katılırsa yapay entropy oluşabilir; NO-OP için bu parçaları maskeli hesaplamak daha doğru olur.
- `source_mask` tamamen boşsa bile NO-OP her zaman açık kaldığı için policy geçerli dağılım üretir.
- `very_neg = torch.finfo(dtype).min` finite olduğu için testlerde `isfinite` kontrolü geçer; softmax sonrasında pratik olarak sıfır olasılık verir.
- `edge_proj` her head için farklı bias öğrenir; bazı head'ler yakın hedeflere, bazıları tehdit yönüne veya global token ilişkisine uzmanlaşabilir.

## 19. Kısa Kod Haritası

| `model.py` parçası | Görev |
| --- | --- |
| `BiasedMultiHeadAttention` | QKV attention + edge bias + padding mask |
| `TransformerBlock` | Pre-norm attention ve FFN residual bloğu |
| `PolicyOutput` | Actor/critic çıktılarını isimli taşıyıcıda toplar |
| `OrbitWarsPolicy.__init__` | Projeksiyonlar, encoder blokları ve head'leri kurar |
| `_build_tokens` | Planet/fleet/global feature'larını token dizisine çevirir |
| `_build_attn_bias` | `(B,T,T,E)` edge feature'ı `(B,H,T,T)` bias'a çevirir |
| `_build_padding_mask` | Gerçek token / padding ayrımını attention maskesine çevirir |
| `forward` | Encoder, actor head'leri ve value head'i çalıştırır |
| `act` | Logitlerden aksiyon örnekler ve log-prob döndürür |
| `pack_observations` | Observation listesini batch tensor dict'e çevirir |